# Task 3 — Part A: Pool Ball Detection

**Goal:** Train an object detector that can locate every ball on a pool table, draw a bounding box around it, and label it with its category (`Black`, `Cue`, `Dot`, `Solid`, `Striped`).

**What we will build:**
1. Explore and understand the dataset (images + COCO annotations)
2. Convert the dataset to YOLO format (what the models expect)
3. Fine-tune **YOLOv8-s** — a fast, CNN-based single-stage detector
4. Fine-tune **RT-DETR-l** — a modern Transformer-based detector
5. Compare both models on standard detection metrics (mAP, precision, recall, speed)

> **Why these two?** They share the exact same Ultralytics training API, making a fair side-by-side comparison trivial to implement. The key difference is architectural: YOLO uses convolutional feature pyramids, while RT-DETR uses Transformer attention — exactly the CNN vs. Transformer narrative the "extra" comparison requires.

---
## 0. Install Dependencies

We only need one library: **Ultralytics**, which packages YOLOv8 and RT-DETR under a single, clean API. It also handles data augmentation, logging, and evaluation automatically.

In [ ]:
# Install Ultralytics (includes YOLOv8 and RT-DETR)
# Run this only once — comment it out after the first run to save time
!pip install ultralytics --quiet

In [ ]:
import os
import json
import shutil
import random
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from ultralytics import YOLO, RTDETR

# Reproducibility
random.seed(42)
np.random.seed(42)

print("All imports successful!")

---
## 1. Data Exploration

Before touching a model, always explore your data. We want to answer:
- How many images and annotations do we have?
- What categories exist, and how are they distributed?
- What do the images and ground-truth boxes look like?

Our dataset lives in `./train/` and uses **COCO JSON format**. In COCO:
- `images` → list of image metadata (id, file_name, width, height)
- `categories` → list of class names and their integer ids
- `annotations` → list of bounding boxes, each linked to an image and a category

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
TRAIN_DIR   = Path("./train")
COCO_JSON   = TRAIN_DIR / "_annotations.coco.json"  # adjust if your filename differs

# ── Load the COCO JSON ─────────────────────────────────────────────────────
with open(COCO_JSON) as f:
    coco = json.load(f)

print(f"Total images      : {len(coco['images'])}")
print(f"Total annotations : {len(coco['annotations'])}")
print(f"\nCategories:")
for cat in coco["categories"]:
    print(f"  id={cat['id']:>3}  name={cat['name']}")

In [ ]:
# ── Build helper look-up dictionaries ──────────────────────────────────────
# Map category id → name  (we will need this later)
id_to_name = {cat["id"]: cat["name"] for cat in coco["categories"]}

# Map image id → filename  (so we can load images by annotation)
id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}

# ── Count annotations per category ─────────────────────────────────────────
# We skip the parent "balls" category (supercategory) if present —
# it is a grouping label, not a real detection class.
real_cat_ids = {cat["id"] for cat in coco["categories"] if cat["name"] != "balls"}

cat_counts = Counter(
    id_to_name[ann["category_id"]]
    for ann in coco["annotations"]
    if ann["category_id"] in real_cat_ids
)

print("Annotation counts per class:")
for name, count in sorted(cat_counts.items(), key=lambda x: -x[1]):
    print(f"  {name:<10}: {count}")

# ── Bar chart ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(cat_counts.keys(), cat_counts.values(), color="steelblue", edgecolor="white")
ax.set_title("Number of annotations per ball category")
ax.set_ylabel("Count")
ax.set_xlabel("Category")
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualise ground-truth boxes on a few sample images ────────────────────
# This is the single most important sanity check: confirm your annotations
# actually sit on the correct objects before spending time training.

# Build a dict: image_id → list of annotations
ann_by_img = {}
for ann in coco["annotations"]:
    if ann["category_id"] not in real_cat_ids:
        continue
    ann_by_img.setdefault(ann["image_id"], []).append(ann)

# Colour palette for each class
PALETTE = {
    "Black"  : "#111111",
    "Cue"    : "#f5f5f5",
    "Dot"    : "#e63946",
    "Solid"  : "#2196f3",
    "Striped": "#ff9800",
}

sample_ids = random.sample(list(ann_by_img.keys()), k=4)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, img_id in zip(axes.flat, sample_ids):
    img_path = TRAIN_DIR / id_to_file[img_id]
    img_bgr  = cv2.imread(str(img_path))
    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)

    for ann in ann_by_img[img_id]:
        x, y, w, h = ann["bbox"]          # COCO bbox: [x_min, y_min, width, height]
        cat_name   = id_to_name[ann["category_id"]]
        color      = PALETTE.get(cat_name, "lime")
        rect = patches.Rectangle(
            (x, y), w, h,
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(x, y - 4, cat_name, color=color, fontsize=8,
                bbox=dict(facecolor="black", alpha=0.4, pad=1))

    ax.set_title(f"Image id {img_id}", fontsize=10)
    ax.axis("off")

plt.suptitle("Ground-truth bounding boxes (sample)", fontsize=13)
plt.tight_layout()
plt.show()

---
## 2. Data Preparation: COCO → YOLO Format

Ultralytics models expect data in **YOLO format**, which is different from COCO:

| Format | Annotation file | Box representation |
|--------|----------------|--------------------|
| COCO   | One big JSON   | `[x_min, y_min, width, height]` in pixels |
| YOLO   | One `.txt` per image | `[class_id, x_center, y_center, width, height]` **normalised 0–1** |

We also need to **create train/val/test splits** ourselves (our dataset has no pre-made splits).

**Split strategy:** 70% train / 15% val / 15% test, stratified at the image level.

In [ ]:
# ── 2.1  Map category name → 0-indexed YOLO class id ───────────────────────
# YOLO class ids must start at 0 and be contiguous.
# We sort alphabetically for reproducibility.
CLASS_NAMES  = sorted([cat["name"] for cat in coco["categories"]
                       if cat["name"] != "balls"])
name_to_yolo = {name: idx for idx, name in enumerate(CLASS_NAMES)}
print("YOLO class mapping:", name_to_yolo)

In [ ]:
# ── 2.2  Create the directory structure expected by Ultralytics ─────────────
#
#   dataset/
#   ├── images/
#   │   ├── train/   ← image files (.jpg)
#   │   ├── val/
#   │   └── test/
#   └── labels/
#       ├── train/   ← one .txt per image
#       ├── val/
#       └── test/

DATASET_DIR = Path("./dataset")

for split in ["train", "val", "test"]:
    (DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

print("Directory structure created.")

In [ ]:
# ── 2.3  Split image ids into train / val / test ────────────────────────────
# Only keep images that have at least one real annotation.
all_img_ids = sorted(ann_by_img.keys())
random.shuffle(all_img_ids)

n       = len(all_img_ids)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
# Test gets the remainder

splits = {
    "train": all_img_ids[:n_train],
    "val"  : all_img_ids[n_train : n_train + n_val],
    "test" : all_img_ids[n_train + n_val :],
}

for split, ids in splits.items():
    print(f"{split:>5}: {len(ids):>4} images")

In [ ]:
# ── 2.4  Convert annotations and copy images ────────────────────────────────
# For each image:
#   1. Copy the .jpg to dataset/images/<split>/
#   2. Write a .txt file to dataset/labels/<split>/
#      Each line: <class_id> <x_center> <y_center> <width> <height>  (all 0-1)

# Build a quick lookup: image_id → (width, height)
img_dims = {img["id"]: (img["width"], img["height"]) for img in coco["images"]}

def coco_bbox_to_yolo(bbox, img_w, img_h):
    """Convert COCO [x_min, y_min, w, h] to YOLO [xc, yc, w, h] normalised."""
    x_min, y_min, w, h = bbox
    x_center = (x_min + w / 2) / img_w
    y_center = (y_min + h / 2) / img_h
    return x_center, y_center, w / img_w, h / img_h

for split, img_ids in splits.items():
    for img_id in img_ids:
        img_info  = next(img for img in coco["images"] if img["id"] == img_id)
        fname     = img_info["file_name"]
        img_w, img_h = img_dims[img_id]

        # 1. Copy image
        src = TRAIN_DIR / fname
        dst = DATASET_DIR / "images" / split / fname
        shutil.copy(src, dst)

        # 2. Write label file
        label_path = DATASET_DIR / "labels" / split / (Path(fname).stem + ".txt")
        lines = []
        for ann in ann_by_img.get(img_id, []):
            cat_name = id_to_name[ann["category_id"]]
            class_id = name_to_yolo[cat_name]
            xc, yc, w, h = coco_bbox_to_yolo(ann["bbox"], img_w, img_h)
            # Clamp values to [0, 1] to handle any annotation boundary errors
            xc = max(0.0, min(1.0, xc))
            yc = max(0.0, min(1.0, yc))
            w  = max(0.0, min(1.0, w))
            h  = max(0.0, min(1.0, h))
            lines.append(f"{class_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        label_path.write_text("\n".join(lines))

print("Dataset converted and organised successfully!")

In [ ]:
# ── 2.5  Write the YAML config file ────────────────────────────────────────
# Ultralytics reads a .yaml file to know where the data lives and
# how many classes exist. This is the only configuration file we need.

yaml_content = f"""# Pool Ball Detection Dataset
path: {DATASET_DIR.resolve()}  # absolute path to dataset root

train: images/train
val:   images/val
test:  images/test

nc: {len(CLASS_NAMES)}  # number of classes
names: {CLASS_NAMES}
"""

yaml_path = Path("pool_balls.yaml")
yaml_path.write_text(yaml_content)
print(yaml_path.read_text())

---
## 3. Model Training

### What is fine-tuning?

Instead of training from random weights (which would need thousands of images), we start from a model that was **pre-trained on COCO** (a large general-purpose detection dataset). We then continue training it on our small pool-ball dataset. The model already knows how to detect edges, shapes, and objects in general — we just teach it to specialise.

### YOLOv8-s — CNN-based detector

YOLOv8 divides the image into a grid. Each grid cell predicts bounding boxes and class probabilities simultaneously in a single forward pass — hence *single-stage*. It uses a **convolutional feature pyramid** to detect objects at multiple scales (useful since pool balls appear at very different sizes depending on camera distance).

### RT-DETR-l — Transformer-based detector

RT-DETR uses a **Transformer encoder** to model global relationships between all image patches. This means a ball in the corner of the image can "attend" to context across the whole scene. Unlike YOLO, it does **not use Non-Maximum Suppression (NMS)** — it directly outputs a fixed set of predictions, which makes its output cleaner but convergence slower.

> **Note:** Both models share the exact same `.train()` API — the only difference is which class you instantiate.

In [ ]:
# ── Training hyperparameters ────────────────────────────────────────────────
# These are conservative, Colab-friendly values.
# Increase epochs (up to 100) if you have more compute budget.

EPOCHS    = 50    # number of full passes over the training set
IMG_SIZE  = 640   # resize all images to 640×640 (standard for YOLO)
BATCH     = 16    # images per gradient update (reduce to 8 if you get OOM errors)
DATA_YAML = str(yaml_path.resolve())

print(f"Training config: epochs={EPOCHS}, imgsz={IMG_SIZE}, batch={BATCH}")

In [ ]:
# ── 3.1  Train YOLOv8-s ─────────────────────────────────────────────────────
# "yolov8s.pt" downloads the small variant pre-trained on COCO.
# The "s" = small — fast enough for Colab free tier.

print("=" * 60)
print("Training YOLOv8-s...")
print("=" * 60)

yolo_model = YOLO("yolov8s.pt")

yolo_results = yolo_model.train(
    data    = DATA_YAML,
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH,
    name    = "yolov8s_pool",   # results saved to runs/detect/yolov8s_pool/
    exist_ok= True,
    verbose = False,            # set True for per-epoch logs
)

print("\nYOLOv8-s training complete!")

In [ ]:
# ── 3.2  Train RT-DETR-l ────────────────────────────────────────────────────
# The Transformer model. Same API, different class.
# Use batch=8 instead of 16 — RT-DETR is heavier on memory.

print("=" * 60)
print("Training RT-DETR-l...")
print("=" * 60)

rtdetr_model = RTDETR("rtdetr-l.pt")

rtdetr_results = rtdetr_model.train(
    data    = DATA_YAML,
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = 8,              # smaller batch for RT-DETR (heavier model)
    name    = "rtdetr_pool",  # results saved to runs/detect/rtdetr_pool/
    exist_ok= True,
    verbose = False,
)

print("\nRT-DETR-l training complete!")

---
## 4. Evaluation

We evaluate both models on the **held-out test set** — images the model has never seen.

### Key metrics explained

| Metric | Meaning | Range |
|--------|---------|-------|
| **Precision** | Of all boxes the model predicted, what fraction were correct? | 0–1, higher is better |
| **Recall** | Of all ground-truth boxes, what fraction did the model find? | 0–1, higher is better |
| **mAP@50** | Mean Average Precision at IoU ≥ 0.50 — the standard detection metric | 0–1, higher is better |
| **mAP@50:95** | Stricter: average mAP across IoU thresholds 0.50 to 0.95 | 0–1, higher is better |

**IoU (Intersection over Union)** measures how much a predicted box overlaps the ground-truth box:
- IoU = 1.0 → perfect overlap
- IoU < 0.5 → prediction is too far off to count as correct

**mAP** averages the Average Precision over all classes. It summarises the precision-recall trade-off into a single number.

In [ ]:
# ── 4.1  Load best weights and run evaluation on test set ──────────────────

YOLO_BEST   = Path("runs/detect/yolov8s_pool/weights/best.pt")
RTDETR_BEST = Path("runs/detect/rtdetr_pool/weights/best.pt")

# Load best checkpoints
yolo_eval   = YOLO(str(YOLO_BEST))
rtdetr_eval = RTDETR(str(RTDETR_BEST))

# Evaluate on the test split
print("Evaluating YOLOv8-s on test set...")
yolo_metrics = yolo_eval.val(data=DATA_YAML, split="test")

print("\nEvaluating RT-DETR-l on test set...")
rtdetr_metrics = rtdetr_eval.val(data=DATA_YAML, split="test")

In [ ]:
# ── 4.2  Build the comparison table ────────────────────────────────────────
# Ultralytics stores metrics in a results object.
# We extract the key numbers and display them side by side.

def extract_metrics(metrics):
    """Pull the core numbers out of an Ultralytics val result object."""
    return {
        "Precision"   : round(float(metrics.box.mp),   4),
        "Recall"      : round(float(metrics.box.mr),   4),
        "mAP@50"      : round(float(metrics.box.map50), 4),
        "mAP@50:95"   : round(float(metrics.box.map),  4),
    }

yolo_m   = extract_metrics(yolo_metrics)
rtdetr_m = extract_metrics(rtdetr_metrics)

print(f"{'Metric':<15} {'YOLOv8-s':>12} {'RT-DETR-l':>12}")
print("-" * 42)
for metric in ["Precision", "Recall", "mAP@50", "mAP@50:95"]:
    print(f"{metric:<15} {yolo_m[metric]:>12.4f} {rtdetr_m[metric]:>12.4f}")

In [ ]:
# ── 4.3  Inference speed comparison ────────────────────────────────────────
# Speed matters for real-time applications.
# We time each model on a single test image (CPU inference to be hardware-agnostic).

import time

test_images = sorted((DATASET_DIR / "images" / "test").glob("*.jpg"))
sample_img  = str(test_images[0])
N_RUNS      = 20  # average over multiple runs for stability

def measure_speed(model, img_path, n_runs=20):
    # Warm-up pass
    model.predict(img_path, verbose=False)
    start = time.perf_counter()
    for _ in range(n_runs):
        model.predict(img_path, verbose=False)
    elapsed = time.perf_counter() - start
    return (elapsed / n_runs) * 1000  # ms per image

yolo_ms   = measure_speed(yolo_eval,   sample_img)
rtdetr_ms = measure_speed(rtdetr_eval, sample_img)

print(f"YOLOv8-s  inference speed : {yolo_ms:.1f} ms/image")
print(f"RT-DETR-l inference speed : {rtdetr_ms:.1f} ms/image")

In [ ]:
# ── 4.4  Visual comparison bar chart ───────────────────────────────────────

metrics_to_plot = ["Precision", "Recall", "mAP@50", "mAP@50:95"]
yolo_vals   = [yolo_m[m]   for m in metrics_to_plot]
rtdetr_vals = [rtdetr_m[m] for m in metrics_to_plot]

x     = np.arange(len(metrics_to_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, yolo_vals,   width, label="YOLOv8-s",  color="#2196f3")
bars2 = ax.bar(x + width/2, rtdetr_vals, width, label="RT-DETR-l", color="#ff9800")

ax.set_ylabel("Score")
ax.set_title("YOLOv8-s vs RT-DETR-l — Detection Metrics on Test Set")
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1.05)
ax.legend()
ax.bar_label(bars1, fmt="%.3f", padding=2, fontsize=8)
ax.bar_label(bars2, fmt="%.3f", padding=2, fontsize=8)

plt.tight_layout()
plt.show()

---
## 5. Qualitative Results

Numbers tell part of the story. Visuals tell the rest. We now run both models on the same set of test images and display their predictions side by side.

**What to look for:**
- Are the boxes tight or loose?
- Are any balls missed (false negatives)?
- Are there ghost detections on the table cloth (false positives)?
- Which model handles overlapping or partially occluded balls better?

In [ ]:
# ── Visualise predictions on 4 test images for each model ──────────────────

CONF_THRESHOLD = 0.25   # only show boxes with confidence ≥ 25%

sample_test = random.sample(test_images, k=4)

fig, axes = plt.subplots(4, 2, figsize=(16, 22))

for row, img_path in enumerate(sample_test):
    for col, (model, model_name) in enumerate([
        (yolo_eval,   "YOLOv8-s"),
        (rtdetr_eval, "RT-DETR-l")
    ]):
        results = model.predict(str(img_path), conf=CONF_THRESHOLD, verbose=False)
        result  = results[0]

        # result.plot() returns a BGR numpy array with boxes drawn
        plotted = result.plot()   # BGR
        plotted = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)

        axes[row, col].imshow(plotted)
        n_det = len(result.boxes)
        axes[row, col].set_title(f"{model_name} — {n_det} detections", fontsize=10)
        axes[row, col].axis("off")

plt.suptitle("Predictions on test images (Left: YOLOv8-s | Right: RT-DETR-l)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 6. Training Curves

Ultralytics saves a `results.csv` in each run folder with per-epoch metrics. Plotting the training curves helps us understand:
- Did the model converge? (loss going down smoothly)
- Is there overfitting? (train mAP rising but val mAP plateauing or dropping)
- Could we have stopped earlier? (early stopping opportunities)

In [ ]:
import pandas as pd

def plot_training_curves(run_dir, model_name, color):
    csv_path = Path(run_dir) / "results.csv"
    if not csv_path.exists():
        print(f"results.csv not found at {csv_path}")
        return

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()   # strip whitespace from headers

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"{model_name} — Training Curves", fontsize=13)

    # Box loss (train vs val)
    axes[0].plot(df["train/box_loss"], label="Train",      color=color)
    axes[0].plot(df["val/box_loss"],   label="Val",  ls="--", color=color, alpha=0.7)
    axes[0].set_title("Box Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    # Class loss
    axes[1].plot(df["train/cls_loss"], label="Train",      color=color)
    axes[1].plot(df["val/cls_loss"],   label="Val",  ls="--", color=color, alpha=0.7)
    axes[1].set_title("Class Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    # mAP@50
    if "metrics/mAP50(B)" in df.columns:
        axes[2].plot(df["metrics/mAP50(B)"], color=color)
        axes[2].set_title("Val mAP@50")
        axes[2].set_xlabel("Epoch")

    plt.tight_layout()
    plt.show()

plot_training_curves("runs/detect/yolov8s_pool",  "YOLOv8-s",  "#2196f3")
plot_training_curves("runs/detect/rtdetr_pool",   "RT-DETR-l", "#ff9800")

---
## 7. Per-Class Analysis

Global mAP hides per-class behaviour. A model might score 0.90 overall by performing very well on common classes while completely missing rare ones. Pool balls are naturally imbalanced: there are many Solids and Stripeds but only one Cue and one Black ball per game.

In [ ]:
# Ultralytics stores per-class AP in metrics.box.ap_class_index and metrics.box.ap
# (available after calling .val())

def per_class_ap(metrics, class_names):
    """Return a dict {class_name: AP@50} from a val result object."""
    ap50_per_class = metrics.box.ap50   # array of AP@50 per class
    return {name: round(float(ap), 4)
            for name, ap in zip(class_names, ap50_per_class)}

yolo_per_class   = per_class_ap(yolo_metrics,   CLASS_NAMES)
rtdetr_per_class = per_class_ap(rtdetr_metrics, CLASS_NAMES)

print(f"{'Class':<10} {'YOLO AP@50':>12} {'RTDETR AP@50':>14}")
print("-" * 40)
for cls in CLASS_NAMES:
    print(f"{cls:<10} {yolo_per_class[cls]:>12.4f} {rtdetr_per_class[cls]:>14.4f}")

In [ ]:
# ── Per-class bar chart ─────────────────────────────────────────────────────
x     = np.arange(len(CLASS_NAMES))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, [yolo_per_class[c]   for c in CLASS_NAMES], width,
       label="YOLOv8-s",  color="#2196f3")
ax.bar(x + width/2, [rtdetr_per_class[c] for c in CLASS_NAMES], width,
       label="RT-DETR-l", color="#ff9800")

ax.set_ylabel("AP@50")
ax.set_title("Per-class AP@50: YOLOv8-s vs RT-DETR-l")
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylim(0, 1.1)
ax.legend()
plt.tight_layout()
plt.show()

---
## 8. Summary & Report Notes

Run this cell to print a clean summary you can copy into your report.

In [ ]:
print("=" * 55)
print("  TASK 3A — BALL DETECTION — RESULTS SUMMARY")
print("=" * 55)

print(f"\nDataset: {n} images  "
      f"(train={len(splits['train'])}, "
      f"val={len(splits['val'])}, "
      f"test={len(splits['test'])})")
print(f"Classes ({len(CLASS_NAMES)}): {', '.join(CLASS_NAMES)}")

print("\n--- Quantitative results (test set) ---")
print(f"{'Metric':<15} {'YOLOv8-s':>12} {'RT-DETR-l':>12}")
print("-" * 42)
for metric in ["Precision", "Recall", "mAP@50", "mAP@50:95"]:
    print(f"{metric:<15} {yolo_m[metric]:>12.4f} {rtdetr_m[metric]:>12.4f}")
print(f"{'Speed (ms/img)':<15} {yolo_ms:>12.1f} {rtdetr_ms:>12.1f}")

print("\n--- Per-class AP@50 ---")
print(f"{'Class':<10} {'YOLOv8-s':>10} {'RT-DETR-l':>12}")
print("-" * 35)
for cls in CLASS_NAMES:
    print(f"{cls:<10} {yolo_per_class[cls]:>10.4f} {rtdetr_per_class[cls]:>12.4f}")

print("\n" + "=" * 55)

---
## 9. Discussion Guide

Use this section to help you write the report and prepare for the presentation.

### Architecture comparison: CNN (YOLO) vs Transformer (RT-DETR)

| Aspect | YOLOv8-s | RT-DETR-l |
|--------|----------|-----------|
| **Core mechanism** | Convolutional feature pyramid | Transformer encoder + CNN backbone |
| **Receptive field** | Local (grows with depth) | Global (attention attends to all patches) |
| **Post-processing** | Requires NMS to remove duplicate boxes | No NMS needed |
| **Convergence** | Fast (~20–30 epochs on small data) | Slower (benefits from more epochs) |
| **Speed** | Faster inference | Slower due to attention computation |
| **Small datasets** | Better (less data hungry) | Worse (Transformers need more data) |

### Likely observations and how to explain them

- **YOLO likely wins on speed** — convolutional operations are highly optimised and parallelise well on GPUs.
- **RT-DETR may score slightly higher on mAP@50:95** — global attention helps with precise box localisation.
- **Cue ball likely has highest AP** — it is white and distinctive; harder classes are Dot (visually similar to Solid from a distance).
- **Class imbalance** — if Dot has few annotations it will have lower AP; mention this in the report.

### Limitations and future work

- **Small dataset (247 images):** both models rely heavily on COCO pre-training; collecting more pool images would improve results.
- **Single angle:** if the dataset only contains one camera angle, the model may generalise poorly to other viewpoints.
- **Augmentation:** increasing augmentation strength (mosaic, random perspective) could further regularise the models.
- **Larger model variants:** YOLOv8-m or RT-DETR-x would likely score higher at the cost of speed.

---

# Task 3 — Part B: Table Retrieval

**Goal:** Given a query pool table image, retrieve the most visually similar image from the gallery (training set).

We implement and compare **three methods** in increasing order of sophistication, following the approach from the lab:

| Method | Feature | Metric | Requires training? |
|--------|---------|--------|--------------------|
| **M1 — MSE (pixel-level)** | Raw pixels (resized) | Mean Squared Error | No |
| **M2 — Pretrained ResNet-50** | 2048-d global avg-pool | MSE on features | No |
| **M3 — Contrastive ResNet-50** | 2048-d trained features | MSE on features | Yes (~10 min on GPU) |

**Key dataset insight:** Images with suffixes `a`, `t`, or `f` in their filename (e.g. `10a_png…`, `10t_png…`) are **different camera views of the same pool table**. Method 3 exploits this by treating them as *positive pairs* in contrastive learning — making the model invariant to viewpoint changes.

**Evaluation:** Sanity check (a slightly translated query should always be more similar to itself than to any retrieved image) + qualitative top-5 retrieval results (good and bad cases).


---
## 10. Data Setup

The retrieval **gallery** is the training partition (207 images). The **queries** are the 20 held-out test images. We load both from `partition.csv` — the lab's official split file.


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import torch
import torchvision
import torchvision.transforms.v2 as T
import torchvision.transforms.functional as TF
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import os, random, re
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
from PIL import Image
from skimage.metrics import mean_squared_error, structural_similarity

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# ── Paths ───────────────────────────────────────────────────────────────────
# Adjust DATASET_DIR to where your dataset lives in Colab
DATASET_DIR   = Path("/content/repo/dataset")
PARTITION_CSV = DATASET_DIR / "partition.csv"

# Build a filename→path index so we find images regardless of subfolder layout
# (works for both flat 'images/' and Roboflow's 'images/train/', 'images/val/' etc.)
IMAGES_DIR = DATASET_DIR / "images"
file_index = {p.name: p for p in IMAGES_DIR.rglob("*.jpg")}
print(f"Indexed {len(file_index)} images under {IMAGES_DIR}")

# ── ImageNet-standard transforms ────────────────────────────────────────────
val_transform = T.Compose([
    T.ToImage(),
    T.Resize((256, 256)),
    T.CenterCrop((224, 224)),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

aug_transform = T.Compose([
    T.ToImage(),
    T.Resize((256, 256)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.3, contrast=0.2, saturation=0.2, hue=0.05),
    T.RandomAffine(degrees=5, translate=(0.05, 0.05)),
    T.CenterCrop((224, 224)),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ── Denormalisation helper for display ──────────────────────────────────────
def denorm(t):
    """Convert a normalised CHW tensor → HWC numpy array for plt.imshow."""
    t = t.permute(1, 2, 0)
    t = t * torch.tensor([0.229, 0.224, 0.225]) + torch.tensor([0.485, 0.456, 0.406])
    return t.clamp(0, 1).numpy()


In [ ]:
# ── Dataset class (matching the lab's PoolDataset) ─────────────────────────
class PoolDataset(Dataset):
    """
    Loads images using partition.csv splits.
    partition: 'train', 'valid', or 'test'
    """
    def __init__(self, partition_csv, partition, transform=None):
        df = pd.read_csv(partition_csv)
        mask = df['partition'] == partition
        self.files = df.loc[mask, 'image_name'].values
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        path = file_index[self.files[i]]
        img  = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

    def get_path(self, i):
        return file_index[self.files[i]]


# ── Create gallery (train) and query (test) datasets ────────────────────────
retrieval_pool = PoolDataset(PARTITION_CSV, 'train', val_transform)
query_data     = PoolDataset(PARTITION_CSV, 'test',  val_transform)

print(f"Gallery (train): {len(retrieval_pool)} images")
print(f"Queries (test) : {len(query_data)} images")

# Preview the first query image
query_image = query_data[0]
fig, ax = plt.subplots(1, 1, figsize=(4, 3))
ax.imshow(denorm(query_image))
ax.set_title("Example query image (test[0])")
ax.axis('off')
plt.tight_layout()
plt.show()


---
## 11. Method 1 — Pixel-Level MSE Retrieval

The simplest baseline: compare images directly at the pixel level using **Mean Squared Error**. Lower MSE = more similar.

**Expected failure mode:** MSE is dominated by the overall image pose and lighting. It will tend to retrieve images with the same camera angle rather than the same ball layout.


In [ ]:
# ── Build MSE similarity scores against the gallery ─────────────────────────
query_image = query_data[0]   # use the first test image as a running example
query_np    = query_image.numpy()

similarities_mse = np.zeros(len(retrieval_pool))
for i in range(len(retrieval_pool)):
    gallery_np          = retrieval_pool[i].numpy()
    similarities_mse[i] = mean_squared_error(query_np, gallery_np)

# Lowest MSE = most similar → argsort ascending
top5_idx_mse = np.argsort(similarities_mse)[:5]

# ── Display ─────────────────────────────────────────────────────────────────
f, axarr = plt.subplots(2, 3, figsize=(14, 8))
f.suptitle("Method 1: Pixel MSE — top-5 retrievals for test[0]", fontsize=12, fontweight='bold')

axarr[0][0].imshow(denorm(query_image))
axarr[0][0].set_title("QUERY", fontweight='bold')
axarr[0][0].axis('off')

for j in range(5):
    idx = top5_idx_mse[j]
    ax  = axarr[(j + 1) // 3][(j + 1) % 3]
    ax.imshow(denorm(retrieval_pool[idx]))
    ax.set_title(f"#{j+1}  MSE = {similarities_mse[idx]:.4f}")
    ax.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# ── Sanity check: query vs translated self vs top-1 retrieved ────────────────
# A good retrieval method should consider the query MORE similar to its own
# slightly-translated version than to any other image in the gallery.

# Apply a 5-pixel horizontal translation to the query
translate_query = TF.affine(
    query_image, angle=0, translate=[5, 0], scale=1.0, shear=[0, 0]
)

sim_trans_mse = mean_squared_error(query_np, translate_query.numpy())
sim_top1_mse  = similarities_mse[top5_idx_mse[0]]

f, axarr = plt.subplots(1, 3, figsize=(12, 4))
f.suptitle("Sanity Check — Method 1 (MSE)", fontsize=11)

axarr[0].imshow(denorm(query_image))
axarr[0].set_title("Query")
axarr[0].axis('off')

axarr[1].imshow(denorm(translate_query))
axarr[1].set_title(f"Translated self\nMSE = {sim_trans_mse:.4f}", color='green')
axarr[1].axis('off')

axarr[2].imshow(denorm(retrieval_pool[top5_idx_mse[0]]))
axarr[2].set_title(f"Top-1 retrieved\nMSE = {sim_top1_mse:.4f}", color='red')
axarr[2].axis('off')

plt.tight_layout()
plt.show()

passed = "✅ PASSED" if sim_trans_mse < sim_top1_mse else "❌ FAILED"
print(f"Sanity check {passed}")
print(f"  Translation MSE : {sim_trans_mse:.6f}")
print(f"  Top-1 gallery   : {sim_top1_mse:.6f}")
print()
print("Interpretation: pixel MSE retrieves images with the same camera pose, not the same game state.")


---
## 12. Method 2 — Pretrained ResNet-50 (No Fine-Tuning)

We use a ResNet-50 pretrained on ImageNet as a **frozen** feature extractor. The global average-pool output (2048-d) captures higher-level semantics than raw pixels, but the model has never seen pool tables specifically.

The similarity metric is again **MSE** on the feature vectors (lower = more similar).


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ── Frozen pretrained ResNet-50 ──────────────────────────────────────────────
class FeatureExtractor(torch.nn.Module):
    """ResNet-50 backbone with the FC head removed. Output: (B, 2048)."""
    def __init__(self, trainable=False):
        super().__init__()
        backbone = torchvision.models.resnet50(
            weights=torchvision.models.ResNet50_Weights.DEFAULT
        )
        self.encoder = torch.nn.Sequential(*list(backbone.children())[:-1])
        for p in self.encoder.parameters():
            p.requires_grad = trainable

    def forward(self, x):
        return self.encoder(x).flatten(1)   # (B, 2048)


pretrained_model = FeatureExtractor(trainable=False).to(device)
pretrained_model.eval()

# ── Extract features for the whole gallery ───────────────────────────────────
@torch.no_grad()
def build_feature_db(model, dataset):
    feats = []
    for i in range(len(dataset)):
        img  = dataset[i].unsqueeze(0).to(device)
        feat = model(img).squeeze(0).cpu()
        feats.append(feat)
    return torch.stack(feats)   # (N, 2048)

print("Extracting gallery features (pretrained) …")
gallery_feats_pretrained = build_feature_db(pretrained_model, retrieval_pool)
print(f"Done — feature matrix: {tuple(gallery_feats_pretrained.shape)}")


In [ ]:
# ── Retrieve top-5 using MSE on pretrained features ──────────────────────────
@torch.no_grad()
def retrieve_top5(query_idx, gallery_feats, model, query_dataset, k=5):
    """Return (top-k indices, distance array) for a query image."""
    q_img  = query_dataset[query_idx].unsqueeze(0).to(device)
    q_feat = model(q_img).squeeze(0).cpu()
    dists  = F.mse_loss(
        q_feat.unsqueeze(0).expand(len(gallery_feats), -1),
        gallery_feats, reduction='none'
    ).mean(dim=1).numpy()
    return np.argsort(dists)[:k], dists


top5_idx_pre, dists_pre = retrieve_top5(0, gallery_feats_pretrained, pretrained_model, query_data)

f, axarr = plt.subplots(2, 3, figsize=(14, 8))
f.suptitle("Method 2: Pretrained ResNet-50 — top-5 retrievals for test[0]", fontsize=12, fontweight='bold')

axarr[0][0].imshow(denorm(query_data[0]))
axarr[0][0].set_title("QUERY", fontweight='bold')
axarr[0][0].axis('off')

for j in range(5):
    idx = top5_idx_pre[j]
    ax  = axarr[(j + 1) // 3][(j + 1) % 3]
    ax.imshow(denorm(retrieval_pool[idx]))
    ax.set_title(f"#{j+1}  dist = {dists_pre[idx]:.4f}")
    ax.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# ── Sanity check — pretrained ResNet ────────────────────────────────────────
@torch.no_grad()
def sanity_check(query_idx, gallery_feats, model, query_dataset, method_name):
    q_img    = query_dataset[query_idx]
    q_moved  = TF.affine(q_img, angle=0, translate=[5, 0], scale=1.0, shear=[0, 0])

    q_feat   = model(q_img.unsqueeze(0).to(device)).squeeze(0).cpu()
    m_feat   = model(q_moved.unsqueeze(0).to(device)).squeeze(0).cpu()

    top1_idx, dists = retrieve_top5(query_idx, gallery_feats, model, query_dataset, k=1)
    sim_trans  = F.mse_loss(q_feat, m_feat).item()
    sim_top1   = dists[top1_idx[0]]

    f, axarr = plt.subplots(1, 3, figsize=(12, 4))
    f.suptitle(f"Sanity Check — {method_name}", fontsize=11)

    axarr[0].imshow(denorm(q_img))
    axarr[0].set_title("Query")
    axarr[0].axis('off')
    axarr[1].imshow(denorm(q_moved))
    axarr[1].set_title(f"Translated self\ndist = {sim_trans:.6f}", color='green')
    axarr[1].axis('off')
    axarr[2].imshow(denorm(retrieval_pool[top1_idx[0]]))
    axarr[2].set_title(f"Top-1 retrieved\ndist = {sim_top1:.6f}", color='red')
    axarr[2].axis('off')

    plt.tight_layout()
    plt.show()

    status = "✅ PASSED" if sim_trans < sim_top1 else "❌ FAILED"
    print(f"  {status}  |  Translation: {sim_trans:.6f}  |  Top-1 gallery: {sim_top1:.6f}")
    return sim_trans < sim_top1

sanity_check(0, gallery_feats_pretrained, pretrained_model, query_data, "Method 2: Pretrained ResNet-50")


---
## 13. Method 3 — Contrastive Learning with Multi-View Positive Pairs

### The Key Insight: Multi-View Images

Looking at the dataset filenames, images with `a`, `t`, or `f` appended to the table number (e.g. `10a_png…`, `10t_png…`, `10_png…`) are **different camera views of the same pool table state**. These represent:
- The same balls in the same positions, shot from different angles
- They should be retrieved as maximally similar to each other

### Contrastive Learning Strategy

We train the ResNet-50 backbone with a **triplet loss**:

| Sample | Meaning |
|--------|---------|
| **Query** | An anchor training image (no augmentation) |
| **Positive** | Another view of the same table (a/t/f sibling) — or an augmented version if no sibling exists |
| **Negative** | An image from a *different* table (always different table ID) |

**Loss:**
```
L = MSE(query, positive) + max(0, margin − MSE(query, negative))
```

This pushes embeddings of the same table together and pulls different tables apart (capped at `margin` to avoid infinitely sparse space).


In [ ]:
# ── Helper: extract table ID from filename ───────────────────────────────────
def get_table_id(filename):
    """
    '10a_png.rf.xxx.jpg' → '10'   (strip trailing a/t/f view indicator)
    '10_png.rf.xxx.jpg'  → '10'   (no suffix, base table)
    '0_png.rf.xxx.jpg'   → '0'
    """
    base = filename.split('_png')[0]          # '10a'
    return re.sub(r'[atf]$', '', base)       # '10'


# ── Triplet Dataset ───────────────────────────────────────────────────────────
class TripletPoolDataset(Dataset):
    """
    Returns (query, positive, negative) triplets.
    Positive = a real multi-view sibling image if one exists in the split,
               otherwise an augmented version of the same image.
    Negative = an image from a definitively different table ID.
    """
    def __init__(self, partition_csv, partition, transform, val_transform):
        df         = pd.read_csv(partition_csv)
        mask       = df['partition'] == partition
        self.files = df.loc[mask, 'image_name'].values
        self.transform     = transform
        self.val_transform = val_transform

        # Group indices by table ID
        self.table_to_idx = defaultdict(list)
        for i, fname in enumerate(self.files):
            tid = get_table_id(fname)
            self.table_to_idx[tid].append(i)

        self.table_ids = list(self.table_to_idx.keys())

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        fname    = self.files[i]
        table_id = get_table_id(fname)
        img      = Image.open(file_index[fname]).convert('RGB')

        # ── Query: no augmentation (standard val transform)
        image_query = self.val_transform(img)

        # ── Positive: real multi-view sibling if available, else augment
        siblings = [j for j in self.table_to_idx[table_id] if j != i]
        if siblings:
            pos_fname = self.files[random.choice(siblings)]
            pos_img   = Image.open(file_index[pos_fname]).convert('RGB')
            image_pos = self.val_transform(pos_img)   # real view — no extra aug needed
        else:
            image_pos = self.transform(img)            # synthetic augmented positive

        # ── Negative: random image from a different table
        other_tables = [t for t in self.table_ids if t != table_id]
        neg_table    = random.choice(other_tables)
        neg_idx      = random.choice(self.table_to_idx[neg_table])
        neg_fname    = self.files[neg_idx]
        neg_img      = Image.open(file_index[neg_fname]).convert('RGB')
        image_neg    = self.val_transform(neg_img)

        return image_query, image_pos, image_neg


# ── Quick sanity: how many tables have multiple views? ──────────────────────
df_all   = pd.read_csv(PARTITION_CSV)
all_ids  = df_all['image_name'].apply(get_table_id)
id_counts = all_ids.value_counts()
multi    = (id_counts > 1).sum()
print(f"Total unique table IDs : {len(id_counts)}")
print(f"Tables with 2+ views   : {multi} ({100*multi/len(id_counts):.0f}%)")
print(f"Example multi-view IDs : {id_counts[id_counts > 1].head(5).to_dict()}")


In [ ]:
# ── Build datasets and dataloaders for contrastive training ─────────────────
train_triplet = TripletPoolDataset(PARTITION_CSV, 'train', aug_transform, val_transform)
val_triplet   = TripletPoolDataset(PARTITION_CSV, 'valid', aug_transform, val_transform)

BATCH_SIZE  = 8
NUM_WORKERS = 2

train_loader = DataLoader(train_triplet, BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_triplet,   BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train triplets : {len(train_triplet)}  ({len(train_loader)} batches)")
print(f"Val   triplets : {len(val_triplet)}  ({len(val_loader)} batches)")


In [ ]:
# ── Contrastive Feature Extractor (trainable) ───────────────────────────────
contrastive_model = FeatureExtractor(trainable=True).to(device)

optimizer  = torch.optim.AdamW(contrastive_model.parameters(), lr=1e-4, weight_decay=1e-4)
loss_pixel = torch.nn.MSELoss(reduction='none')
MARGIN     = 1.0
EPOCHS     = 20


def one_epoch_contrastive(model, optimizer, dataloader, is_training):
    """One pass of triplet contrastive loss (MSE-based)."""
    model.train() if is_training else model.eval()
    avg_loss = 0.0
    ctx = torch.enable_grad() if is_training else torch.no_grad()
    with ctx:
        for img_q, img_p, img_n in dataloader:
            img_q = img_q.to(device)
            img_p = img_p.to(device)
            img_n = img_n.to(device)

            rep_q = model(img_q)
            rep_p = model(img_p)
            rep_n = model(img_n)

            # Minimise distance to positive (same table)
            loss = loss_pixel(rep_q, rep_p).mean()
            # Maximise distance to negative (different table), clipped at margin
            neg_dist = loss_pixel(rep_q, rep_n).mean(dim=1)
            loss += torch.mean(torch.clamp(MARGIN - neg_dist, min=0))

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            avg_loss += loss.item() / len(dataloader)
    return avg_loss


print("Starting contrastive training …")
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    tr_loss = one_epoch_contrastive(contrastive_model, optimizer, train_loader, True)
    vl_loss = one_epoch_contrastive(contrastive_model, optimizer, val_loader,   False)
    train_losses.append(tr_loss)
    val_losses.append(vl_loss)
    print(f"Epoch {epoch+1:2d}/{EPOCHS} — train loss: {tr_loss:.4f}  val loss: {vl_loss:.4f}")

# ── Plot training curves ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label='Train', color='#e63946')
ax.plot(val_losses,   label='Val',   color='#2196f3')
ax.set_xlabel('Epoch')
ax.set_ylabel('Contrastive Loss')
ax.set_title('Method 3: Contrastive Training')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── Build gallery features with the contrastive model ───────────────────────
print("Extracting gallery features (contrastive model) …")
gallery_feats_contrastive = build_feature_db(contrastive_model, retrieval_pool)
print(f"Done — feature matrix: {tuple(gallery_feats_contrastive.shape)}")

# ── Retrieve top-5 for the same example query ────────────────────────────────
top5_idx_ctr, dists_ctr = retrieve_top5(0, gallery_feats_contrastive,
                                         contrastive_model, query_data)

f, axarr = plt.subplots(2, 3, figsize=(14, 8))
f.suptitle("Method 3: Contrastive ResNet-50 — top-5 retrievals for test[0]",
           fontsize=12, fontweight='bold')

axarr[0][0].imshow(denorm(query_data[0]))
axarr[0][0].set_title("QUERY", fontweight='bold')
axarr[0][0].axis('off')

for j in range(5):
    idx = top5_idx_ctr[j]
    ax  = axarr[(j + 1) // 3][(j + 1) % 3]
    ax.imshow(denorm(retrieval_pool[idx]))
    ax.set_title(f"#{j+1}  dist = {dists_ctr[idx]:.4f}")
    ax.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# ── Sanity check — contrastive model ────────────────────────────────────────
sanity_check(0, gallery_feats_contrastive, contrastive_model, query_data,
             "Method 3: Contrastive ResNet-50")


---
## 14. Qualitative Evaluation

We run all three methods on the full test set and display results side by side.  
Layout per query:
- **Row 0:** Method 1 (pixel MSE)
- **Row 1:** Method 2 (pretrained ResNet-50)
- **Row 2:** Method 3 (contrastive ResNet-50)

Red/orange/blue borders distinguish the methods.


In [ ]:
def show_three_method_comparison(query_idx, n_results=3):
    """
    Display top-n_results for all three methods for one query image.
    """
    q_img    = query_data[query_idx]
    q_np     = q_img.numpy()

    # ── Method 1 (MSE pixel) ──────────────────────────────────────────
    dists_m1 = np.array([
        mean_squared_error(q_np, retrieval_pool[i].numpy())
        for i in range(len(retrieval_pool))
    ])
    top_m1 = np.argsort(dists_m1)[:n_results]

    # ── Method 2 (pretrained) ─────────────────────────────────────────
    top_m2, dists_m2 = retrieve_top5(query_idx, gallery_feats_pretrained,
                                      pretrained_model, query_data, k=n_results)

    # ── Method 3 (contrastive) ────────────────────────────────────────
    top_m3, dists_m3 = retrieve_top5(query_idx, gallery_feats_contrastive,
                                      contrastive_model, query_data, k=n_results)

    # ── Plot ──────────────────────────────────────────────────────────
    n_rows, n_cols = 3, n_results + 1
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows))
    fname = query_data.files[query_idx]
    fig.suptitle(f"Query: {fname}", fontsize=11, fontweight='bold', y=1.01)

    methods   = ['M1: Pixel MSE', 'M2: Pretrained', 'M3: Contrastive']
    top_lists = [top_m1, top_m2, top_m3]
    dist_maps = [dists_m1, dists_m2, dists_m3]
    colors    = ['#e63946', '#ff9800', '#2196f3']

    for row, (mname, tops, dists, color) in enumerate(
            zip(methods, top_lists, dist_maps, colors)):
        # Query column
        axes[row, 0].imshow(denorm(q_img))
        axes[row, 0].set_ylabel(mname, fontsize=9)
        if row == 0:
            axes[row, 0].set_title("QUERY", fontweight='bold', fontsize=10)
        axes[row, 0].axis('off')

        # Retrieved columns
        for col, idx in enumerate(tops):
            ax = axes[row, col + 1]
            ax.imshow(denorm(retrieval_pool[idx]))
            ax.set_title(f"#{col+1}  {dists[idx]:.4f}", fontsize=8, color=color)
            ax.axis('off')
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_edgecolor(color)
                spine.set_linewidth(2.5)

    plt.tight_layout()
    plt.show()


# ── Run for 5 random test images ─────────────────────────────────────────────
N_SHOW = min(5, len(query_data))
chosen = random.sample(range(len(query_data)), N_SHOW)

print(f"Showing retrieval results for {N_SHOW} query images …\n")
for qi in chosen:
    show_three_method_comparison(qi, n_results=3)


### 14.1 Failure Case Analysis

We automatically find the query where Method 3 (contrastive) has the **highest** top-1 distance — its hardest case — and examine why it fails.

**Typical failure modes:**
- **Nearly-empty tables:** When there are very few balls, the felt texture dominates all three methods' features, making very different game states look identical.
- **Unusual view angles:** An extreme angle that doesn't match any training view makes the contrastive model uncertain.


In [ ]:
# ── Find the worst retrieval query for Method 3 ─────────────────────────────
worst_qi, worst_dist = None, -1.0
for qi in range(len(query_data)):
    _, d = retrieve_top5(qi, gallery_feats_contrastive, contrastive_model,
                          query_data, k=1)
    top1_dist = d[np.argmin(d)]
    if top1_dist > worst_dist:
        worst_dist = top1_dist
        worst_qi   = qi

print(f"Failure candidate: query index {worst_qi} ({query_data.files[worst_qi]})")
print(f"Contrastive top-1 distance: {worst_dist:.4f}\n")

show_three_method_comparison(worst_qi, n_results=3)

# ── Detailed failure analysis ─────────────────────────────────────────────
q_img = query_data[worst_qi]

top1_m1 = np.argsort(np.array([
    mean_squared_error(q_img.numpy(), retrieval_pool[i].numpy())
    for i in range(len(retrieval_pool))
]))[0]
top1_m2, dm2 = retrieve_top5(worst_qi, gallery_feats_pretrained,  pretrained_model,  query_data, k=1)
top1_m3, dm3 = retrieve_top5(worst_qi, gallery_feats_contrastive, contrastive_model, query_data, k=1)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("Failure Case — Enlarged View", fontsize=12, fontweight='bold')

axes[0].imshow(denorm(q_img))
axes[0].set_title("Query", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(denorm(retrieval_pool[top1_m1]))
axes[1].set_title("M1 top-1\n(pixel MSE)", color='#e63946')
axes[1].axis('off')

axes[2].imshow(denorm(retrieval_pool[top1_m2[0]]))
axes[2].set_title(f"M2 top-1\n(pretrained, dist={dm2[top1_m2[0]]:.4f})", color='#ff9800')
axes[2].axis('off')

axes[3].imshow(denorm(retrieval_pool[top1_m3[0]]))
axes[3].set_title(f"M3 top-1\n(contrastive, dist={dm3[top1_m3[0]]:.4f})", color='#2196f3')
axes[3].axis('off')

plt.tight_layout()
plt.show()


### 14.2 Sanity Check Across All Test Queries

We run the translation sanity check across **all 20 test images** for each method. A higher pass rate means the method better understands that a slightly transformed version of an image is more similar to itself than to any other image.


In [ ]:
results = {'M1_pixel': [], 'M2_pretrained': [], 'M3_contrastive': []}

for qi in range(len(query_data)):
    q_img   = query_data[qi]
    q_moved = TF.affine(q_img, angle=0, translate=[5, 0], scale=1.0, shear=[0, 0])
    q_np    = q_img.numpy()

    # Method 1
    sim_trans = mean_squared_error(q_np, q_moved.numpy())
    dists_all = np.array([mean_squared_error(q_np, retrieval_pool[i].numpy())
                          for i in range(len(retrieval_pool))])
    results['M1_pixel'].append(sim_trans < dists_all.min())

    # Method 2
    with torch.no_grad():
        q_feat  = pretrained_model(q_img.unsqueeze(0).to(device)).squeeze(0).cpu()
        m_feat  = pretrained_model(q_moved.unsqueeze(0).to(device)).squeeze(0).cpu()
    sim_trans2 = F.mse_loss(q_feat, m_feat).item()
    top1_dist2 = F.mse_loss(q_feat.unsqueeze(0).expand(len(gallery_feats_pretrained), -1),
                             gallery_feats_pretrained, reduction='none').mean(1).min().item()
    results['M2_pretrained'].append(sim_trans2 < top1_dist2)

    # Method 3
    with torch.no_grad():
        q_feat3 = contrastive_model(q_img.unsqueeze(0).to(device)).squeeze(0).cpu()
        m_feat3 = contrastive_model(q_moved.unsqueeze(0).to(device)).squeeze(0).cpu()
    sim_trans3 = F.mse_loss(q_feat3, m_feat3).item()
    top1_dist3 = F.mse_loss(q_feat3.unsqueeze(0).expand(len(gallery_feats_contrastive), -1),
                              gallery_feats_contrastive, reduction='none').mean(1).min().item()
    results['M3_contrastive'].append(sim_trans3 < top1_dist3)


print("=" * 50)
print("  SANITY CHECK PASS RATE (all 20 test queries)")
print("=" * 50)
for name, passed in results.items():
    rate = 100 * sum(passed) / len(passed)
    bar  = "█" * int(rate / 5)
    print(f"  {name:18s}: {sum(passed):2d}/{len(passed)} ({rate:.0f}%) {bar}")


---
## 15. Part B — Summary & Report Notes

### Method Comparison

| | M1: Pixel MSE | M2: Pretrained ResNet | M3: Contrastive ResNet |
|---|---|---|---|
| **Feature** | Raw pixels (224×224) | 2048-d GAP (ImageNet) | 2048-d GAP (fine-tuned) |
| **Training** | None | None | ~20 epochs (triplet loss) |
| **Positive pairs** | — | — | Multi-view a/t/f siblings |
| **Metric** | MSE on pixels | MSE on features | MSE on features |
| **Captures** | Pixel intensity / pose | ImageNet semantics | Pool table semantics |
| **Main failure** | Retrieves same camera pose | Not domain-adapted | Few-ball empty tables |

### Key Takeaways

1. **Pixel MSE** is dominated by camera pose — all retrieved images tend to look like the query's viewpoint, not its game state. Sanity check usually fails.

2. **Pretrained ResNet** already improves over pixels by capturing higher-level structure (number of balls, distribution on the table) without any domain training.

3. **Contrastive learning with multi-view pairs** is the strongest method. By telling the model "these three images of table 10 (10_, 10a_, 10t_) should have the same embedding", the model becomes viewpoint-invariant and focuses on the game state (which balls are where) rather than the camera angle.

4. **The a/t/f hint** from the dataset is the critical feature. Without it, contrastive learning would only use synthetic augmentation as positives, which is much weaker.


In [ ]:
# ── Print final summary ──────────────────────────────────────────────────────
print("=" * 60)
print("  TASK 3B — TABLE RETRIEVAL — FINAL SUMMARY")
print("=" * 60)
print(f"\n  Gallery size : {len(retrieval_pool)} images  (train partition)")
print(f"  Query count  : {len(query_data)} images  (test partition)")

df_part = pd.read_csv(PARTITION_CSV)
n_multi = (df_part['image_name'].apply(get_table_id).value_counts() > 1).sum()
print(f"  Multi-view tables (a/t/f) used as positives: {n_multi}")

print("\n  Sanity check pass rate:")
for name, passed in results.items():
    rate = 100 * sum(passed) / len(passed)
    print(f"    {name:20s}: {rate:.0f}%")

print("\n" + "=" * 60)
